In [6]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from neuralhydrology.evaluation.metrics import calculate_metrics

from scipy import signal
from neuralhydrology.datautils import utils
from neuralhydrology.evaluation.metrics import _validate_inputs, _mask_valid

In [2]:
# ------------------- Paths -------------------
RUN_DIR = Path("./runs")

ensemble_metrics_dir=Path("./ensemble_peak_metrics")
ensemble_metrics_dir.mkdir(exist_ok=True)

In [3]:
run_pattern = "camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*"
# run_pattern = "total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed*"

matched_paths = sorted(RUN_DIR.glob(f"{run_pattern}/test/model_epoch030/test_results.p"))
print(f"Found {len(matched_paths)} runs: {[p.parts[-4] for p in matched_paths]}")

Found 8 runs: ['camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed111_2204_062451', 'camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed222_2204_175151', 'camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed333_2204_230424', 'camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed444_2304_041450', 'camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed555_2304_092510', 'camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed666_2304_143622', 'camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed777_2404_014030', 'camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed888_2404_065130']


In [5]:
# Load all runs
all_runs_data = []
for file_path in matched_paths:
    with open(file_path, "rb") as f:
        all_runs_data.append(pickle.load(f))

# Average the simulated flows across seeds, per basin
ensemble_data = {}

for basin_id in all_runs_data[0].keys():
    # Stack simulated flows from all seeds: shape (n_seeds, n_timesteps, time_step)
    sims = np.stack([
        run[basin_id]['1D']['xr']['streamflow_sim'].values #['QObs(mm/d)_sim'].values 
        for run in all_runs_data
    ], axis=0)
    
    mean_sim = np.mean(sims, axis=0)  # Average across seeds
    
    # Copy structure from first run, replace sim with ensemble mean
    xr_ensemble = all_runs_data[0][basin_id]['1D']['xr'].copy()
    xr_ensemble['streamflow_sim'].values[:] = mean_sim #['QObs(mm/d)_sim'].values[:] = mean_sim #
    
    ensemble_data[basin_id] = {'1D': {'xr': xr_ensemble}}

ensemble_data

{'camels_01022500': {'1D': {'xr': <xarray.Dataset> Size: 58kB
   Dimensions:         (date: 3652, time_step: 1)
   Coordinates:
     * date            (date) datetime64[ns] 29kB 2009-10-01 ... 2019-09-30
     * time_step       (time_step) int64 8B 0
   Data variables:
       streamflow_obs  (date, time_step) float32 15kB 1.01 0.85 0.73 ... nan nan
       streamflow_sim  (date, time_step) float32 15kB 1.024 0.8396 ... nan nan}},
 'camels_01031500': {'1D': {'xr': <xarray.Dataset> Size: 58kB
   Dimensions:         (date: 3652, time_step: 1)
   Coordinates:
     * date            (date) datetime64[ns] 29kB 2009-10-01 ... 2019-09-30
     * time_step       (time_step) int64 8B 0
   Data variables:
       streamflow_obs  (date, time_step) float32 15kB 0.38 0.32 0.27 ... nan nan
       streamflow_sim  (date, time_step) float32 15kB 0.4248 0.3707 ... nan nan}},
 'camels_01047000': {'1D': {'xr': <xarray.Dataset> Size: 58kB
   Dimensions:         (date: 3652, time_step: 1)
   Coordinates:
     * 

In [7]:
from scipy import signal
import numpy as np
import pandas as pd
from neuralhydrology.datautils import utils
from neuralhydrology.evaluation.metrics import _validate_inputs, _mask_valid

def custom_mean_peak_timing(obs, sim, window=None, resolution='1D', datetime_coord=None, distance=100):
    _validate_inputs(obs, sim)
    obs, sim = _mask_valid(obs, sim)

    peaks, _ = signal.find_peaks(obs.values, distance=distance, prominence=np.std(obs.values))

    if datetime_coord is None:
        datetime_coord = utils.infer_datetime_coord(obs)
    if window is None:
        window = max(int(utils.get_frequency_factor('12h', resolution)), 3)

    timing_errors = []
    for idx in peaks:
        if (idx - window < 0) or (idx + window >= len(obs)) or (
            pd.date_range(obs[idx - window][datetime_coord].values,
                          obs[idx + window][datetime_coord].values,
                          freq=resolution).size != 2 * window + 1):
            continue

        if (sim[idx] > sim[idx - 1]) and (sim[idx] > sim[idx + 1]):
            peak_sim = sim[idx]
        else:
            values = sim[idx - window:idx + window + 1]
            peak_sim = values[values.argmax()]

        peak_obs = obs[idx]
        delta = peak_obs.coords[datetime_coord] - peak_sim.coords[datetime_coord]
        timing_errors.append(np.abs(delta.values / pd.to_timedelta(resolution)))

    return np.mean(timing_errors) if len(timing_errors) > 0 else np.nan


def custom_missed_peaks(obs, sim, window=None, resolution='1D', percentile=80, datetime_coord=None, distance=30):
    _validate_inputs(obs, sim)
    obs, sim = _mask_valid(obs, sim)

    min_obs_height = np.percentile(obs.values, percentile)
    min_sim_height = np.percentile(sim.values, percentile)

    peaks_obs_times, _ = signal.find_peaks(obs, distance=distance, height=min_obs_height)
    peaks_sim_times, _ = signal.find_peaks(sim, distance=distance, height=min_sim_height)

    if len(peaks_obs_times) == 0:
        return 0.

    if datetime_coord is None:
        datetime_coord = utils.infer_datetime_coord(obs)
    if window is None:
        window = max(int(utils.get_frequency_factor('12h', resolution)), 1)

    missed_events = 0
    for idx in peaks_obs_times:
        if (idx - window < 0) or (idx + window >= len(obs)) or (
            pd.date_range(obs[idx - window][datetime_coord].values,
                          obs[idx + window][datetime_coord].values,
                          freq=resolution).size != 2 * window + 1):
            continue

        nearby_peak_sim_index = np.where(np.abs(peaks_sim_times - idx) <= window)[0]
        if len(nearby_peak_sim_index) == 0:
            missed_events += 1

    return missed_events / len(peaks_obs_times)

In [8]:
PEAK_TIMING_DISTANCE = 50
MISSED_PEAKS_DISTANCE = 15
WINDOW = 5

In [10]:
all_metrics = {}
for basin_id, basin_data in ensemble_data.items():
    xr_ds = basin_data['1D']['xr'].isel(time_step=0)
    
    obs = xr_ds['streamflow_obs']
    sim = xr_ds['streamflow_sim']
    
    if obs.isnull().all() or sim.isnull().all():
        print(f"Skipping {basin_id} — all observed/simulated values are NaN")
        continue
    
    all_metrics[basin_id] = {
        'Peak-Timing': custom_mean_peak_timing(obs, sim, window=WINDOW, distance=PEAK_TIMING_DISTANCE, resolution="1D", datetime_coord="date"),
        'Missed-Peaks': custom_missed_peaks(obs, sim, window=WINDOW, distance=MISSED_PEAKS_DISTANCE, resolution="1D", datetime_coord="date")
    }

df_metrics = pd.DataFrame(all_metrics).T
df_metrics.index.name = 'basin_id'
df_metrics

Skipping camels_06291500 — all observed/simulated values are NaN


,Peak-Timing,Missed-Peaks
basin_id,,
camels_01022500,0.272727,0.152174
camels_01031500,0.105263,0.130435
camels_01047000,0.136364,0.127273
camels_01052500,0.080000,0.087719
camels_01054200,0.318182,0.157895
...,...,...
camels_14309500,0.500000,0.277778
camels_14316700,0.400000,0.222222
camels_14325000,0.133333,0.219512
